In [1]:
!nvidia-smi

Mon Jul 13 16:35:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
!pip install ninja

In [ ]:
cuda_source = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <float.h>
#include <vector>

#define BLOCK_Q 32
#define BLOCK_K 32
#define MAX_D 128

__global__ void flash_atten_fwd_kernel(
    const float* __restrict__ Q,
    const float* __restrict__ K,
    const float* __restrict__ V,
    float* __restrict__ O,
    float* __restrict__ L,
    const int N,
    const int d,
    const float scale,
    const bool causal
) {
    extern __shared__ float shared_mem[];
    float* Kt = shared_mem; 
    float* Vt = shared_mem + BLOCK_K * d;

    const int bh = blockIdx.y;
    const float* Q_block = Q + (size_t)bh * N * d;
    const float* K_block = K + (size_t)bh * N * d;
    const float* V_block = V + (size_t)bh * N * d;
    float* O_block = O + (size_t)bh * N * d;
    float* L_block = L + (size_t)bh * N;

    const int q_block_start = blockIdx.x * BLOCK_Q;
    const int q_block_end = min(q_block_start + BLOCK_Q - 1, N - 1);
    const int tid = threadIdx.x;
    const int q_row = q_block_start + tid;

    float row_max = -FLT_MAX;
    float row_sum = 0.0f;

    float acc[MAX_D];
    float q_req[MAX_D];

    for (int i = 0; i < d; i++) {
        acc[i] = 0.0f;
    }

    if (q_row < N) {
        for (int i = 0; i < d; i++) {
            q_req[i] = Q_block[q_row * d + i];
        }
    }

    int k_loop_end;
    if (causal) {
        k_loop_end = q_block_end + 1;
    } else {
        k_loop_end = N;
    }

    for (int k_block_start = 0; k_block_start < k_loop_end; k_block_start += BLOCK_K) {
        int k_rows = min(BLOCK_K, k_loop_end - k_block_start);

        for (int r = tid; r < k_rows; r += BLOCK_Q) {
            for (int i = 0; i < d; i++) {
                Kt[r * d + i] = K_block[(k_block_start + r) * d + i];
                Vt[r * d + i] = V_block[(k_block_start + r) * d + i];
            }
        }
        __syncthreads();

        if (q_row < N) {
            const bool causal_mask = causal && (k_block_start + k_rows - 1 > q_block_start);
            float scores[BLOCK_K];
            float block_max = -FLT_MAX;

            for (int i = 0; i < k_rows; i++) {
                int k_row = k_block_start + i;
                if (causal_mask && k_row > q_row) {
                    scores[i] = -FLT_MAX;
                    continue;
                }
                float s = 0.0f;
                for (int j = 0; j < d; j++) {
                    s += q_req[j] * Kt[i * d + j];
                }
                s *= scale;
                scores[i] = s;
                block_max = fmaxf(block_max, s);
            }

            if (block_max > -FLT_MAX) {
                float new_max = fmaxf(row_max, block_max);
                float correction = expf(row_max - new_max);

                row_sum *= correction;
                for (int i = 0; i < d; i++) {
                    acc[i] *= correction;
                }

                for (int i = 0; i < k_rows; i++) {
                    if (scores[i] == -FLT_MAX) continue;
                    float p = expf(scores[i] - new_max);
                    row_sum += p;
                    for (int j = 0; j < d; j++) {
                        acc[j] += p * Vt[i * d + j];
                    }
                }
                row_max = new_max;
            }
        }
        __syncthreads();
    }

    if (q_row < N) {
        for (int i = 0; i < d; i++) {
            O_block[q_row * d + i] = acc[i] / row_sum;
        }
        L_block[q_row] = row_max + logf(row_sum);
    }
}

torch::Tensor flash_attn_forward(
    torch::Tensor Q,
    torch::Tensor K,
    torch::Tensor V
) {
    const int BH = 1;
    const int N = Q.size(0);
    const int d = Q.size(1);

    const float scale = 1.0f / sqrtf((float)d);
    const bool causal = false;

    auto O = torch::empty({N, d}, Q.options());
    auto L = torch::empty({N}, Q.options());

    const int num_q_blocks = (N + BLOCK_Q - 1) / BLOCK_Q;
    const dim3 grid(num_q_blocks, BH);
    const int threads_per_block = BLOCK_Q;
    const size_t shared_bytes = 2 * (size_t)BLOCK_K * d * sizeof(float);

    flash_atten_fwd_kernel<<<grid, threads_per_block, shared_bytes>>>(
        Q.data_ptr<float>(), K.data_ptr<float>(), V.data_ptr<float>(),
        O.data_ptr<float>(), L.data_ptr<float>(), N, d, scale, causal
    );

    cudaError_t err = cudaGetLastError();
    TORCH_CHECK(err == cudaSuccess, "CUDA kernel launch failed: ", cudaGetErrorString(err));

    return O;
}
"""

In [5]:
import torch
from torch.utils.cpp_extension import load_inline

cpp_source = "torch::Tensor flash_attn_forward(torch::Tensor Q, torch::Tensor K, torch::Tensor V);"

flash_attn_cuda = load_inline(
    name="flash_attn_cuda",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["flash_attn_forward"],
    verbose=True,
)

In [6]:
def naive_attention(Q, K, V):
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d ** 0.5)
    probs = torch.softmax(scores, dim=-1)
    return probs @ V

torch.manual_seed(0)
device = "cuda"

N, d = 300, 64
Q = torch.randn(N, d, device=device, dtype=torch.float32)
K = torch.randn(N, d, device=device, dtype=torch.float32)
V = torch.randn(N, d, device=device, dtype=torch.float32)

out_naive = naive_attention(Q, K, V)
out_cuda = flash_attn_cuda.flash_attn_forward(Q, K, V)

max_abs_diff = (out_naive - out_cuda).abs().max().item()
print(f"max abs difference vs naive reference: {max_abs_diff:.3e}")
print("match:", torch.allclose(out_naive, out_cuda, atol=1e-3))

max abs difference vs naive reference: 3.576e-07
match: True
